In [1]:
from openai import OpenAI
base_url="http://localhost:11434/v1"
client=OpenAI(base_url=base_url,api_key="ollama")

In [2]:
system_prompt="""
You are a synthetic data generator.
Return only valid JSON.
No markdown"""

In [3]:
task="music sales data for a music label OCD Records"
temperature=0.7
n_samples=1000

In [4]:
def build_prompt(task,n):
    return f""" Generate {n} synthetic examples for :
    Task: {task}
    Return format:
               [{{
                    "artist": "...",
                    "album": "...",
                    "units_sold": 200,
                    "month": "May",
                    "year": 2023
                    }}]"""

In [5]:
def generate(task,n):
    messages=[{'role':'system','content':system_prompt},
              {'role':'user','content':build_prompt(task,n)}]
    response=client.chat.completions.create(model='qwen-local',messages=messages,temperature=temperature)
    return response.choices[0].message.content

In [6]:
import json
import re
def extract_json(text):
    match=re.search(r"\[.*\]",text,re.DOTALL)
    if not match:
        raise ValueError("No JSON found")
    return json.loads(match.group())

In [7]:
def generate_dataset(task,n_samples,batch_size):
    dataset=[]
    for start in range(0,n_samples,batch_size):
        print(f"Generating {start+batch_size}/{n_samples}")
        raw=generate(task,batch_size)
        batch=extract_json(raw)
        dataset.extend(batch)
    return dataset

In [8]:
data=generate_dataset(task,n_samples=1000,batch_size=25)

Generating 25/1000
Generating 50/1000
Generating 75/1000
Generating 100/1000
Generating 125/1000
Generating 150/1000
Generating 175/1000
Generating 200/1000
Generating 225/1000
Generating 250/1000
Generating 275/1000
Generating 300/1000
Generating 325/1000
Generating 350/1000
Generating 375/1000
Generating 400/1000
Generating 425/1000
Generating 450/1000
Generating 475/1000
Generating 500/1000
Generating 525/1000
Generating 550/1000
Generating 575/1000
Generating 600/1000
Generating 625/1000
Generating 650/1000
Generating 675/1000
Generating 700/1000
Generating 725/1000
Generating 750/1000
Generating 775/1000
Generating 800/1000
Generating 825/1000
Generating 850/1000
Generating 875/1000
Generating 900/1000
Generating 925/1000
Generating 950/1000
Generating 975/1000
Generating 1000/1000


In [9]:
import pandas as pd
df=pd.DataFrame(data)
df

,artist,album,units_sold,month,year,units sold,units
0,The Midnight Echoes,Neon Dreams,1200.0,May,2023,NaN,NaN
1,Electric Dreams,Synthwave Memories,850.0,March,2022,NaN,NaN
2,Soulful Strings,Harmony in Blue,1500.0,July,2023,NaN,NaN
3,Urban Beats,City Pulse,980.0,January,2021,NaN,NaN
4,Pop Vortex,Dancing in the Light,2450.0,September,2023,NaN,NaN
...,...,...,...,...,...,...,...
994,Desert Drifters,Winds of Sand,360.0,September,2024,NaN,NaN
995,Quantum Beats,Timeless Rhythms,160.0,October,2024,NaN,NaN
996,Frostbite Echoes,Winter Whispers,230.0,November,2024,NaN,NaN
997,The Midnight Echoes,Echoes of the Past,315.0,December,2024,NaN,NaN


In [12]:
df=df.drop(['units sold','units'],axis=1)
df

,artist,album,units_sold,month,year
0,The Midnight Echoes,Neon Dreams,1200.0,May,2023
1,Electric Dreams,Synthwave Memories,850.0,March,2022
2,Soulful Strings,Harmony in Blue,1500.0,July,2023
3,Urban Beats,City Pulse,980.0,January,2021
4,Pop Vortex,Dancing in the Light,2450.0,September,2023
...,...,...,...,...,...
994,Desert Drifters,Winds of Sand,360.0,September,2024
995,Quantum Beats,Timeless Rhythms,160.0,October,2024
996,Frostbite Echoes,Winter Whispers,230.0,November,2024
997,The Midnight Echoes,Echoes of the Past,315.0,December,2024


In [13]:
df.to_csv('music_data.csv')